<a href="https://colab.research.google.com/github/bananmagdy3-crypto/Mall-Customer-Segmentation/blob/main/Mall_Customer_Segmentation_KMeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛍️ Mall Customer Segmentation using K-Means Clustering
### A Complete Unsupervised Machine Learning Project (Google Colab Ready)

This notebook segments mall customers into meaningful groups based on their **Age**, **Annual Income**, and **Spending Score**, using **K-Means Clustering** (Unsupervised Learning).

**Why clustering and not classification?**
The dataset has no predefined "correct answer" (target label) telling us which segment a customer belongs to. We don't know the segments in advance — we want the algorithm to *discover* them from the data itself. That is exactly what unsupervised learning (clustering) does. Classification would require pre-labeled customer categories, which we don't have.

---


In [1]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Make plots look clean and professional
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42  # fixed seed so results are reproducible every time you run this notebook

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
import os

filename = "Mall_Customers.csv"

if not os.path.exists(filename):
    print("Dataset not found locally. Please upload 'Mall_Customers.csv' below.")
    from google.colab import files
    uploaded = files.upload()  # opens a file picker in Colab

    # Automatically detect the uploaded CSV file name
    csv_files = [f for f in uploaded.keys() if f.lower().endswith(".csv")]
    if len(csv_files) == 0:
        raise FileNotFoundError("No CSV file was uploaded. Please re-run this cell and upload Mall_Customers.csv")
    filename = csv_files[0]

# Load the dataset into a pandas DataFrame
df = pd.read_csv(filename)
print(f"Loaded file: {filename}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")


Dataset not found locally. Please upload 'Mall_Customers.csv' below.


In [ ]:
# First 5 rows
df.head()


In [ ]:
# Last 5 rows
df.tail()


In [ ]:
# Dataset shape (rows, columns)
print("Dataset shape:", df.shape)


In [ ]:
# Column names
print("Column names:", list(df.columns))


In [ ]:
# Data types of each column
df.dtypes


In [ ]:
# Basic statistical summary of numeric columns
df.describe()


In [ ]:
# Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
# Check for duplicated rows
duplicate_count = df.duplicated().sum()
print(f"Number of duplicated rows: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New shape:", df.shape)
else:
    print("No duplicates found. No rows removed.")


In [ ]:
# Standardize column names for easier coding (remove spaces/parentheses)
df.columns = df.columns.str.strip()
rename_map = {}
for col in df.columns:
    if "Annual Income" in col:
        rename_map[col] = "Annual_Income"
    elif "Spending Score" in col:
        rename_map[col] = "Spending_Score"
    elif col.strip() == "Gender" or col.strip() == "Genre":
        rename_map[col] = "Gender"
    elif col.strip() == "Age":
        rename_map[col] = "Age"
    elif col.strip() == "CustomerID":
        rename_map[col] = "CustomerID"

df = df.rename(columns=rename_map)
print("Columns after cleanup:", list(df.columns))


In [ ]:
# Confirm data types are correct (Age, Annual_Income, Spending_Score should be numeric)
df.dtypes


In [ ]:
# Check for outliers in numeric columns using boxplots
numeric_cols = ["Age", "Annual_Income", "Spending_Score"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(f"Outlier Check: {col}")
plt.tight_layout()
plt.show()


In [ ]:
# 1. Gender distribution
plt.figure()
sns.countplot(data=df, x="Gender", hue="Gender", palette="Set2", legend=False)
plt.title("Gender Distribution of Mall Customers")
plt.xlabel("Gender")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# 2. Age distribution
plt.figure()
sns.histplot(df["Age"], bins=15, kde=True, color="teal")
plt.title("Age Distribution of Mall Customers")
plt.xlabel("Age")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# 3. Annual Income distribution
plt.figure()
sns.histplot(df["Annual_Income"], bins=15, kde=True, color="orange")
plt.title("Annual Income Distribution (k$)")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# 4. Spending Score distribution
plt.figure()
sns.histplot(df["Spending_Score"], bins=15, kde=True, color="purple")
plt.title("Spending Score Distribution (1-100)")
plt.xlabel("Spending Score")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# 5. Age vs Annual Income
plt.figure()
sns.scatterplot(data=df, x="Age", y="Annual_Income", hue="Gender", palette="Set1")
plt.title("Age vs Annual Income")
plt.xlabel("Age")
plt.ylabel("Annual Income (k$)")
plt.show()


In [ ]:
# 6. Annual Income vs Spending Score
plt.figure()
sns.scatterplot(data=df, x="Annual_Income", y="Spending_Score", hue="Gender", palette="Set1")
plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.show()


In [ ]:
# 7. Age vs Spending Score
plt.figure()
sns.scatterplot(data=df, x="Age", y="Spending_Score", hue="Gender", palette="Set1")
plt.title("Age vs Spending Score")
plt.xlabel("Age")
plt.ylabel("Spending Score (1-100)")
plt.show()


In [ ]:
# 8. Correlation heatmap for numerical features
plt.figure(figsize=(6, 5))
corr = df[["Age", "Annual_Income", "Spending_Score"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


In [ ]:
# Select the main features for clustering
features_2d = df[["Annual_Income", "Spending_Score"]]

# Standardize the features
scaler = StandardScaler()
scaled_features_2d = scaler.fit_transform(features_2d)

print("Features used for clustering: Annual_Income, Spending_Score")
print("Sample of scaled data:\n", scaled_features_2d[:5])


In [ ]:
# OPTIONAL: features including Age, for comparison later
features_3d = df[["Age", "Annual_Income", "Spending_Score"]]
scaled_features_3d = scaler.fit_transform(features_3d)
print("Optional 3-feature set (Age, Annual_Income, Spending_Score) prepared for later comparison.")


In [ ]:
# Elbow Method: test K = 1 to 10
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans_test = KMeans(n_clusters=k, init="k-means++", random_state=RANDOM_STATE, n_init=10)
    kmeans_test.fit(scaled_features_2d)
    wcss.append(kmeans_test.inertia_)

plt.figure()
plt.plot(list(K_range), wcss, marker="o", color="darkblue")
plt.title("Elbow Method: WCSS vs Number of Clusters (K)")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS (Inertia)")
plt.xticks(list(K_range))
plt.show()


In [ ]:
# Silhouette Score: test K = 2 to 10 (silhouette score is undefined for K=1)
silhouette_scores = []
K_range_sil = range(2, 11)

for k in K_range_sil:
    kmeans_test = KMeans(n_clusters=k, init="k-means++", random_state=RANDOM_STATE, n_init=10)
    labels_test = kmeans_test.fit_predict(scaled_features_2d)
    score = silhouette_score(scaled_features_2d, labels_test)
    silhouette_scores.append(score)

plt.figure()
plt.plot(list(K_range_sil), silhouette_scores, marker="o", color="darkgreen")
plt.title("Silhouette Score vs Number of Clusters (K)")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(list(K_range_sil))
plt.show()

best_k = list(K_range_sil)[int(np.argmax(silhouette_scores))]
print(f"K with highest silhouette score: {best_k}")


In [ ]:
# Set the optimal number of clusters based on the elbow + silhouette analysis above
optimal_k = 5  # Adjust this if your elbow/silhouette plots suggest a different value
print(f"Selected optimal K = {optimal_k}")


In [ ]:
# Build and fit the final K-Means model
kmeans = KMeans(n_clusters=optimal_k, init="k-means++", random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_features_2d)

# Add the cluster label back to the original dataframe
df["Cluster"] = cluster_labels

print("Cluster labels assigned. First 10 rows:")
df.head(10)


In [ ]:
# Cluster distribution: how many customers fall into each cluster
cluster_counts = df["Cluster"].value_counts().sort_index()
print("Number of customers per cluster:")
print(cluster_counts)


In [ ]:
# Professional scatter plot of customer segments with centroids
plt.figure(figsize=(9, 6))

palette = sns.color_palette("Set1", n_colors=optimal_k)
sns.scatterplot(
    data=df, x="Annual_Income", y="Spending_Score",
    hue="Cluster", palette=palette, s=70, edgecolor="black", alpha=0.8
)

# Plot centroids (convert back from scaled space to original space)
centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)
plt.scatter(
    centroids_original[:, 0], centroids_original[:, 1],
    s=300, c="black", marker="X", label="Centroids"
)

plt.title("Mall Customer Segments (K-Means Clustering)", fontsize=15)
plt.xlabel("Annual Income (k$)", fontsize=12)
plt.ylabel("Spending Score (1-100)", fontsize=12)
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Calculate average values and size for each cluster
cluster_summary = df.groupby("Cluster").agg(
    Avg_Age=("Age", "mean"),
    Avg_Annual_Income=("Annual_Income", "mean"),
    Avg_Spending_Score=("Spending_Score", "mean"),
    Num_Customers=("CustomerID", "count")
).round(1)

cluster_summary


In [ ]:
# Assign a meaningful business name to each cluster based on its actual average values
def name_cluster(row):
    income = row["Avg_Annual_Income"]
    spending = row["Avg_Spending_Score"]
    income_median = cluster_summary["Avg_Annual_Income"].median()
    spending_median = cluster_summary["Avg_Spending_Score"].median()

    if income >= income_median and spending >= spending_median:
        return "High Value Customers"
    elif income >= income_median and spending < spending_median:
        return "High Income / Low Spending Customers"
    elif income < income_median and spending >= spending_median:
        return "Potential Customers (Low Income, High Spending)"
    else:
        return "Low Engagement Customers"

cluster_summary["Segment_Name"] = cluster_summary.apply(name_cluster, axis=1)
cluster_summary


In [ ]:
# Fit K-Means using Age + Annual_Income + Spending_Score
kmeans_3d = KMeans(n_clusters=optimal_k, init="k-means++", random_state=RANDOM_STATE, n_init=10)
labels_3d = kmeans_3d.fit_predict(scaled_features_3d)

df["Cluster_with_Age"] = labels_3d

print("Cluster sizes using only Income & Spending Score:")
print(df["Cluster"].value_counts().sort_index())
print("\nCluster sizes using Age + Income & Spending Score:")
print(df["Cluster_with_Age"].value_counts().sort_index())


In [ ]:
# Apply PCA to reduce the 3 features to 2 dimensions for visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_result = pca.fit_transform(scaled_features_3d)

df["PCA1"] = pca_result[:, 0]
df["PCA2"] = pca_result[:, 1]

explained_var = pca.explained_variance_ratio_
print(f"Variance explained by PC1: {explained_var[0]*100:.1f}%")
print(f"Variance explained by PC2: {explained_var[1]*100:.1f}%")
print(f"Total variance retained: {sum(explained_var)*100:.1f}%")

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="PCA1", y="PCA2",
    hue="Cluster_with_Age", palette=sns.color_palette("Set1", n_colors=optimal_k),
    s=70, edgecolor="black", alpha=0.8
)
plt.title("Customer Segments (Age + Income + Spending) via PCA", fontsize=15)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Final evaluation metrics for our main 2-feature clustering model
final_inertia = kmeans.inertia_
final_silhouette = silhouette_score(scaled_features_2d, df["Cluster"])

print(f"Final Model (K={optimal_k}) Evaluation:")
print(f"  Inertia (WCSS): {final_inertia:.2f}")
print(f"  Silhouette Score: {final_silhouette:.3f}")
